# ImageGrains 2.0 Web — Pre-Processing Benchmark (`audit/preprocessing/`)

Benchmarks optional client-side image pre-processing (**CLAHE**, **bilinear upscaling**, **adaptive upscaling**, **gamma/brightness invariance sanity check**, and optional **mild denoise**) using the shipped **TypeScript** `webGrainy` pipeline (`onnxruntime-web@1.31.1`, `prepareWorkingImage(..., "auto")`, `applyCLAHE`, `runUnetCellposeInference` / `runCpsamCellposeInference`, `flowsToMasksBrowser` defaults: `flow=0.45`, `cellprob=0`, `minSize=30`, `maxAreaFraction=0.40`).

- **Models (`inYourOwnBrowser/imagegrains-2.0-onnx` @ `98e76b1374dc22cfff8193aca71c308f2b4dd3b3`):** `ig2_unet_fp32.onnx`, `ig2_cpsam_int8_wo_fp32scale.onnx`, and 5-image `ig2_cpsam_fp16.onnx` spot check.
- **5-tile WASM parity guard:** Python ORT CPU bridge is allowed **only** if `max |WASM - PyCPU| <= BRIDGE_MAX_ABS_TOL` (`0.05`) on 5 tiles; otherwise native `ort_web_wasm` runs every image.
- **Adaptive threshold `T` (Variant F):** Selected strictly on `official_split == "train"`, never on the 40 `test` images.
- **Original-pixel metrics & GT-resolution PQ:** Lengths `/ s`, areas `/ s**2`, predicted label masks resized back to GT resolution via nearest-neighbor.


In [ ]:
# CONFIG — Preprocessing Benchmark (High-RAM CPU Colab supported; GPU not required)
REPO_URL = "https://github.com/LukeAFullard/webGrainy.git"
REPO_BRANCH = "audit/preprocessing-benchmark"
REPO_DIR = "/content/webGrainy"

HF_REPO = "inYourOwnBrowser/imagegrains-2.0-onnx"
HF_REVISION = "98e76b1374dc22cfff8193aca71c308f2b4dd3b3"
MODELS = ["unet_fp32", "cpsam_int8", "cpsam_fp16"]  # cpsam_fp16 automatically runs 5-image spot check
VARIANTS = [
    "A", "B", "C", "D", "E", "F",
    "G_gamma07", "G_gamma14", "G_bright_p20", "G_bright_m20",
    "H_bilateral", "H_median3",
]

MAX_TEST_IMAGES = 40          # 40 = full held-out test split (13 sites)
TRAIN_T_MAX_IMAGES = 203      # Choose T on official_split == "train" for U-Net (203 = all train images)
TRAIN_T_MAX_IMAGES_CPSAM = 13 # Stratified train images for CP-SAM T selection (avoids multi-day WASM)
WASM_NUM_THREADS = 4
BRIDGE_MAX_ABS_TOL = 0.05     # 5-tile max |WASM - PyCPU|; INT8 (~0.4-0.6) falls back to native WASM
SEED = 42

DRIVE_CACHE_DIR = "/content/drive/MyDrive/webGrainy_preprocessing_audit"
DRIVE_DATA_DIR = "/content/drive/MyDrive/imagegrains_datasets"
LOCAL_WORK_DIR = "/content/prep_bench_work"
SKIP_CACHED_ON_DRIVE = True


In [ ]:
# Cell 1: Mount Drive, clone & build webGrainy, install Python packages, verify shipped TS runners
import os, shutil, subprocess, sys

DRIVE_MOUNTED = False
if os.path.exists("/content"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = os.path.isdir("/content/drive/MyDrive")
    except Exception as e:
        print("Drive mount skipped:", e)

os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
if DRIVE_MOUNTED:
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

# Install Node 22+ if needed
need_node = False
try:
    v = subprocess.check_output(["node", "-v"], text=True).strip()
    if int(v.lstrip("v").split(".")[0]) < 20:
        need_node = True
except Exception:
    need_node = True

if need_node:
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs", shell=True, check=True)

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR], check=True)
subprocess.run(["npm", "ci"], cwd=REPO_DIR, check=True)
subprocess.run(["npm", "run", "build"], cwd=REPO_DIR, check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "onnxruntime>=1.19", "opencv-python-headless", "pillow", "tifffile",
    "scikit-image", "scipy", "pandas", "matplotlib", "tqdm"
], check=True)

for rel in [
    "audit/followup/ts_cellpose_runner.mjs",
    "audit/followup/ts_postprocess_runner.mjs",
    "audit/preprocessing/ts_preprocess_worker.mjs",
]:
    subprocess.run(["node", os.path.join(REPO_DIR, rel), "--selftest"], cwd=REPO_DIR, check=True)
print("Node + TS self-tests PASSED. Drive mounted:", DRIVE_MOUNTED)


In [ ]:
# Cell 2: Download & SHA-256 verify HF ONNX models + Zenodo IG2_data.zip (cached on Drive)
import hashlib, urllib.request, zipfile

MODEL_FILES = {
    "unet_fp32": ("ig2_unet_fp32.onnx", "5e692c5593ac15e614b77602e1d4f8c2f84dd939f4386b1f7e5641dbcd62053c"),
    "cpsam_int8": ("ig2_cpsam_int8_wo_fp32scale.onnx", "2cda0be5b81f1d0c0ac36998beb9f741e11833aacd089c4f5cee92488bab6f43"),
    "cpsam_fp16": ("ig2_cpsam_fp16.onnx", "944a114a0cb1a3a215c11affefe7211caac6da2b74895c496d5a7b5a23b6230d"),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

MODELS_DIR = os.path.join(LOCAL_WORK_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)
for mkey in MODELS:
    fname, expected_sha = MODEL_FILES[mkey]
    dst = os.path.join(MODELS_DIR, fname)
    drive_src = os.path.join(DRIVE_DATA_DIR, "hf_models", HF_REVISION, fname) if DRIVE_MOUNTED else ""
    if DRIVE_MOUNTED and os.path.exists(drive_src) and sha256_file(drive_src) == expected_sha:
        shutil.copy2(drive_src, dst)
    if not (os.path.exists(dst) and sha256_file(dst) == expected_sha):
        url = f"https://huggingface.co/{HF_REPO}/resolve/{HF_REVISION}/{fname}"
        print("Downloading", url)
        urllib.request.urlretrieve(url, dst)
        assert sha256_file(dst) == expected_sha, f"SHA mismatch for {fname}"
        if DRIVE_MOUNTED:
            os.makedirs(os.path.dirname(drive_src), exist_ok=True)
            shutil.copy2(dst, drive_src)
    print(f"Verified {fname} ({os.path.getsize(dst)/1e6:.1f} MB)")

IG2_ZIP_URL = "https://zenodo.org/records/17866827/files/IG2_data.zip?download=1"
DATA_DIR = os.path.join(LOCAL_WORK_DIR, "ig2_data")
if not os.path.exists(os.path.join(DATA_DIR, "CT")):
    zip_path = os.path.join(DRIVE_DATA_DIR, "IG2_data.zip") if (DRIVE_MOUNTED and os.path.exists(os.path.join(DRIVE_DATA_DIR, "IG2_data.zip"))) else os.path.join(LOCAL_WORK_DIR, "IG2_data.zip")
    if not os.path.exists(zip_path):
        print("Downloading IG2_data.zip from Zenodo...")
        urllib.request.urlretrieve(IG2_ZIP_URL, zip_path)
    os.makedirs(DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    nested = os.path.join(DATA_DIR, "IG2_data")
    if os.path.isdir(nested):
        for name in os.listdir(nested):
            shutil.move(os.path.join(nested, name), os.path.join(DATA_DIR, name))
        os.rmdir(nested)
assert os.path.isdir(os.path.join(DATA_DIR, "CT", "test")), f"Missing extracted test folder under {DATA_DIR}"
print("IG2_data ready at:", DATA_DIR)


In [ ]:
# Cell 3: Run preprocessing benchmark with per-variant/per-image Drive caching
CACHE_DIR = os.path.join(DRIVE_CACHE_DIR, "cache") if (DRIVE_MOUNTED and SKIP_CACHED_ON_DRIVE) else os.path.join(LOCAL_WORK_DIR, "cache")
OUT_DIR = os.path.join(LOCAL_WORK_DIR, "out")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, "audit/preprocessing/run_benchmark.py"),
    "--data-dir", DATA_DIR,
    "--models-dir", MODELS_DIR,
    "--out-dir", OUT_DIR,
    "--cache-dir", CACHE_DIR,
    "--models", *MODELS,
    "--variants", *VARIANTS,
    "--max-test", str(MAX_TEST_IMAGES),
    "--train-t-max", str(TRAIN_T_MAX_IMAGES),
    "--train-t-max-cpsam", str(TRAIN_T_MAX_IMAGES_CPSAM),
    "--wasm-threads", str(WASM_NUM_THREADS),
    "--bridge-tol", str(BRIDGE_MAX_ABS_TOL),
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

if DRIVE_MOUNTED:
    shutil.copy2(os.path.join(OUT_DIR, "results.json"), os.path.join(DRIVE_CACHE_DIR, "results.json"))
    shutil.copy2(os.path.join(OUT_DIR, "results_per_image.csv"), os.path.join(DRIVE_CACHE_DIR, "results_per_image.csv"))
    print("Synced results.json & results_per_image.csv to", DRIVE_CACHE_DIR)


In [ ]:
# Cell 4: Summary tables, decision rule PASS/FAIL check, per-site table, & charts
import json
import matplotlib.pyplot as plt
import pandas as pd

results = json.load(open(os.path.join(OUT_DIR, "results.json")))
df = pd.read_csv(os.path.join(OUT_DIR, "results_per_image.csv"))

for mkey, mdata in results["models"].items():
    print(f"\n==================== {mkey} ====================")
    if mkey in results.get("adaptive_threshold_train", {}):
        ch = results["adaptive_threshold_train"][mkey]["chosen"]
        print("Train-selected F rule: upscale=%s when working_med_b_px < %.1f px (train n=%d, trig=%d)" % (
            ch["upscale_variant"], ch["T_px"], ch["n_train"], ch["n_triggered"]))
    base_v = mdata["variants"]["A"]
    base_pq = base_v["pq_vs_gt"]["mean"]
    base_d50 = base_v["vs_gt"]["D50"]["median_ape_pct"]
    base_r10 = base_v["size_class_recall"]["lt10"]["pooled_recall"]
    print("A (baseline) | PQ=%.4f | F1=%.4f | MedD50=%.2f%% | recall<10px=%.3f | estWASM=%.1fs" % (
        base_pq, base_v["f1_50_vs_gt"]["mean"], base_d50, base_r10, base_v.get("est_wasm_wall_s_total", float("nan"))))

    for group_name in ("paired_vs_A", "subset_paired_vs_A"):
        if not mdata.get(group_name):
            continue
        print(f"--- {group_name} ---")
        for var, p in mdata[group_name].items():
            v = mdata["variants"][var]
            dr = p["decision_rule"]
            r10 = v["size_class_recall"]["lt10"]["pooled_recall"]
            est_s = v.get("est_wasm_wall_s_total", float("nan"))
            pq_m = v["pq_vs_gt"]["mean"]
            d50_m = v["vs_gt"]["D50"]["median_ape_pct"]
            print(
                "  %-12s | PQ=%.4f (d=%+.4f [%+.4f,%+.4f]) | MedD50=%.2f%% (d=%+.2fpp [%+.2f,%+.2f]) | "
                "r<10=%.3f | estWASM=%.1fs | worst=%s(%+.2fpp) | PASS_DEF=%s" % (
                    var, pq_m, p["mean_dPQ"], p["dPQ_ci95"][0], p["dPQ_ci95"][1],
                    d50_m, p["mean_dAPE_pp"], p["dAPE_ci95_pp"][0], p["dAPE_ci95_pp"][1],
                    r10, est_s, p["worst_site"], p["worst_site_dMed_d50_ape_pp"], dr["pass_default"]
                )
            )

# Plot PQ vs D50 APE trade-off for each full model
for mkey, mdata in results["models"].items():
    vars_plot = [v for v in ("A", "B", "C", "D", "E", "F") if v in mdata["variants"]]
    if len(vars_plot) < 2:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    pqs = [mdata["variants"][v]["pq_vs_gt"]["mean"] for v in vars_plot]
    d50s = [mdata["variants"][v]["vs_gt"]["D50"]["median_ape_pct"] for v in vars_plot]
    r10s = [mdata["variants"][v]["size_class_recall"]["lt10"]["pooled_recall"] for v in vars_plot]
    axes[0].bar(vars_plot, pqs, color="#2b6cb0")
    axes[0].set_title(f"{mkey}: Mean PQ vs GT")
    axes[0].set_ylabel("PQ")
    axes[1].bar(vars_plot, r10s, color="#2f855a")
    axes[1].set_title(f"{mkey}: Small-Grain Recall (<10 px b-axis)")
    axes[1].set_ylabel("Pooled Recall (IoU > 0.5)")
    plt.tight_layout()
    chart_p = os.path.join(OUT_DIR, f"summary_{mkey}.png")
    plt.savefig(chart_p, dpi=150)
    plt.show()
